# line-room · fine-tune Needle 2 on Colab

Trains a LoRA adapter on this room's five tools and emits a `.cact` that the
browser engine loads unchanged.

**Before running:** Runtime → Change runtime type → **T4 GPU**. A 45M model is
tiny for a T4; the whole run is a few minutes. It also works on a CPU runtime,
just slower.

Colab is Linux x86_64, which removes the two worst obstacles from doing this on
an Intel Mac: `jaxlib` has manylinux wheels, so `pip install cactus-needle`
resolves normally. Two platform-independent traps remain and are handled below
— the checkpoint path and the float16 problem.

In [ ]:
!nvidia-smi || echo 'No GPU — this still works, just slower.'

## 1. Install

Colab ships its own jax. Installing `cactus-needle[gpu]` may move it, and Colab
then wants a runtime restart. If the next cell complains about jax, use
Runtime → Restart session and re-run from here — you do **not** need to re-run
this install.

In [ ]:
!pip install -q 'cactus-needle[gpu]'

import jax
print('jax', jax.__version__, '| devices:', jax.devices())

## 2. Base checkpoint

`needle finetune` auto-downloads `checkpoints/needle2.pkl` and gets a 404: the
repo publishes it at `weights/needle2.pkl`. Fetch it explicitly. Note this is a
clean *file-not-found*, not a network error — that is how you tell a wrong path
from a blocked connection.

In [ ]:
!mkdir -p checkpoints
!wget -q --show-progress -O checkpoints/needle2.pkl \
  https://huggingface.co/Cactus-Compute/needle2/resolve/main/weights/needle2.pkl
!ls -lh checkpoints/

## 3. float16 → float32

The published checkpoint is float16 — all 56 tensors. Inference is fine, but
the base forward pass already reaches logits of ~7.6e3 against fp16's 65504
ceiling. LoRA adapters are created with `weight.dtype` and `merge_lora` casts
back, so adapters *and* optimizer state stay fp16 too, and the loss goes `nan`
within about four steps.

This is a numeric-range problem, not a compute problem — a GPU does not fix it.
Export re-quantizes to ~2 bits anyway, so carrying fp32 through training costs
nothing in the end.

In [ ]:
import pickle, numpy as np
from flax.traverse_util import flatten_dict, unflatten_dict

ck = pickle.load(open('checkpoints/needle2.pkl', 'rb'))
flat = flatten_dict(ck['params'])
print('before:', {str(v.dtype) for v in flat.values()})
ck['params'] = unflatten_dict({k: np.asarray(v, np.float32) for k, v in flat.items()})
pickle.dump(ck, open('checkpoints/needle2-fp32.pkl', 'wb'), protocol=4)
print('wrote checkpoints/needle2-fp32.pkl')

## 4. Dataset

Upload `make_dataset.py` and `tools.json` from `line-room/train/`. Nothing else
needs uploading — the data is generated deterministically from a fixed seed, so
the script reproduces the identical 659 rows here.

In [ ]:
from google.colab import files
import os

missing = [f for f in ('make_dataset.py', 'tools.json') if not os.path.exists(f)]
if missing:
    print('upload:', ', '.join(missing))
    files.upload()

!python make_dataset.py

## 5. Train

`--batch-size 32` because a T4 has room the Mac did not; if you land on a CPU
runtime, drop it to 8.

Sequences are median 445 tokens and p95 469, so `--max-len 512` covers the set
and costs half of the 1024 default.

For reference: on a 4-core Intel Mac with no GPU this took roughly 80 minutes
of compute plus a 51-minute compile.

In [ ]:
!needle finetune data/train.jsonl \
  --checkpoint checkpoints/needle2-fp32.pkl \
  --epochs 3 --batch-size 32 --max-len 512 \
  --lora-rank 16 --lora-alpha 32 --lr 1e-4 \
  --out checkpoints/room_lora.pkl

## 6. Export

Merges the adapter into the base weights and re-quantizes.

**Do not pass `--bits 2`.** The shipped model is mixed precision *averaging*
two bits, and that per-tensor map lives in the checkpoint config as
`weight_bits`. `build` uses it automatically — but only when `--bits` is
absent. Passing it overrides the map with uniform 2-bit and quietly produces a
differently-quantized model than the base you mean to compare against.

In [ ]:
!needle build checkpoints/needle2-fp32.pkl \
  --lora checkpoints/room_lora.pkl \
  --out needle2-tuned.cact

!ls -lh needle2-tuned.cact

## 7. Download

Put the file in `line-room/models/`, then compare it against the base through
the real browser engine — see `train/README.md`:

```
http://127.0.0.1:8231/?weights=needle2.cact
http://127.0.0.1:8231/?weights=needle2-tuned.cact
```

Paste `eval.js` into the console and `await lineRoomEval()`. Keep the tuned
weights only if they actually win — and watch the `refusal` bucket, because a
LoRA that pulls everything toward calling a tool regresses it, and that
regression is worse than the failures being fixed.

The browser caches weights by URL in the Cache API, so if you rebuild under the
same filename, clear it first:
`await caches.delete('line-room-models-v1')`.

In [ ]:
from google.colab import files
files.download('needle2-tuned.cact')